[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/34_speculative_decoding_solution.ipynb)

# ✅ Solution: Speculative Decoding

Implement the **acceptance/rejection step** of speculative decoding — a technique for accelerating LLM inference.

### Signature
```python
def speculative_decode(target_probs, draft_probs, draft_tokens) -> list[int]:
    # target_probs: (K, V) from target (large) model
    # draft_probs: (K, V) from draft (small) model
    # draft_tokens: (K,) tokens sampled by draft model
    # Returns: list of accepted tokens (1 to K)
```

### Algorithm
For each position i = 0, ..., K-1:
1. `ratio = target_probs[i, token_i] / draft_probs[i, token_i]`
2. Accept with probability `min(1, ratio)`
3. If rejected: sample from `normalize(max(0, target - draft))`, append, and stop


In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge')
except ImportError:
    pass


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import optax
import math


In [ ]:
# ✅ SOLUTION

import jax.numpy as jnp
def speculative_decode(target_probs,draft_probs,draft_tokens):
    out=[]
    for i,t in enumerate(draft_tokens):
        t=int(t); ratio=target_probs[i,t]/jnp.maximum(draft_probs[i,t],1e-10)
        if ratio>=1: out.append(t); continue
        p=jnp.maximum(target_probs[i]-draft_probs[i],0); out.append(int(jnp.argmax(p if p.sum()>0 else target_probs[i]))); return out
    return out


In [ ]:
# Verify
print(speculative_decode)


In [ ]:
from jax_judge import check
check("speculative_decoding")
